<a href="https://colab.research.google.com/github/Banking-Analytics-Lab/DLinBankingBook/blob/main/Labs/TextBook_Lab_Chap7_Payment_Sequences.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multimodal Behavioral Scoring: Application Data, Payment Histories and the Loan Network

*Deep Learning in Banking* (Bravo, Maldonado and Óskarsdóttir, Wiley, 2026), Chapter 7: updated case study (2026).

> **New case (2026).** This case replaces the case study printed in Chapter 7 of the book. The original case, which combines time series, static variables, images and text in one model, is still available: [open it in Colab](https://colab.research.google.com/github/Banking-Analytics-Lab/DLinBankingBook/blob/main/Labs/TextBook_Lab_Chap7_Multimodal_Data.ipynb) or [view it on GitHub](https://github.com/Banking-Analytics-Lab/DLinBankingBook/blob/main/Labs/TextBook_Lab_Chap7_Multimodal_Data.ipynb).

A behavioral scoring model ranks the loans a bank already holds by their risk of a serious delinquency. Banks build it from **hand-made features** of the payment history: the worst delinquency in the last 12 months, the number of late months, how much of the balance was paid down. Someone had to decide which features to compute.

In this lab each loan has three kinds of data, three **modalities**:

1. a **table**: the application data, one row per loan (credit score, loan-to-value, debt-to-income, and so on);
2. a **sequence**: the last 24 monthly payment records of the loan, six numbers per month;
3. a **graph**: the loan's neighbors, the other loans in the same 3-digit ZIP code, sold by the same lender, or handled by the same servicer. We summarize what the neighbors did each month, so this modality also arrives as a sequence.

We ask two questions. First, **can the raw monthly records go into the model directly, without hand-made features, and do as well?** If they can, the gain is operational. Hand-made features carry a technical debt (Sculley et al., 2015): someone designs them, writes and validates the code, documents it, and keeps it working when the data change. A well-designed data flow, which turns the raw records into sequences, and a multimodal model that reads them, can remove that work. Second, **does the network of loans add information that the loan's own data do not have?** Chapter 5 builds graphs of customers and providers; here the graph enters a multimodal model as one more modality.

We fuse the modalities in three ways: intermediate and late fusion, as defined in the chapter, and a third design that we call residual fusion. We compare every fused model with each modality alone, with a **placebo** for each modality, and with a strong gradient-boosting model that uses the hand-made features. A fused model must beat each modality alone. It needs only to match the hand-made features: then we avoid the technical debt of building and maintaining them.

The data are half of the loans in the Freddie Mac Single-Family Loan-Level Dataset sample files for the origination years 1999 to 2007. The Chapter 3 lab uses the same data source, and its hybrid GRU has the same architecture as model M5 here. The two labs differ in the question (here, hand-made features against raw modalities, and the value of the network), the target (a default event in the next 12 months for a loan that is current today, instead of a delinquency in the next quarter), the population (the loans of the years before the 2008 crisis) and the split (by time, instead of at random). We train on 2002 to 2006 and test on 2008, the year the US mortgage crisis peaked.

This is a **behavioral scoring model**: it ranks loans. We do not calibrate its outputs, so they should not be read as probabilities of default (PDs).

### How this lab follows the chapter

The case study of Chapter 7 has six algorithms (7.1 to 7.6). This lab follows the same steps, but it does not map to them one to one: its three modalities are a table, a payment history and a graph, not an image or a text. The table shows where each algorithm is, and what changes.

| Chapter 7 | In this lab | What changes |
|---|---|---|
| 7.1 Load the data sources, define the target, split | Sections 1 to 3 | Three sources instead of four. The split is by time (train 2002-2006, test 2008), and every held-out set is chosen by loan, not at random with stratification. A random split would put quarters of the same loan, and labels from the test period, into training. |
| 7.2 Preprocess each channel | Sections 1 to 3 and 6 | `PreprocessStaticData` becomes `prep_tab` (missing codes, 0/1 columns, scaling fitted on training rows); `PreprocessTimeSeries` becomes `prep_seq`, and the graph gets `prep_nb`. There is no text or image channel. We use no class weights: we compare models by AUC, which ranks and does not depend on a threshold. |
| 7.3 Integrate the features, build data loaders | Sections 3 and 6 | The table and the two sequences are joined by loan and quarter. The data fit on the GPU, so we index mini-batches directly instead of writing a `Dataset` class. |
| 7.4 Define the models | Sections 6 to 9 | The static extractor is the same MLP. The time-series extractor is a GRU instead of a transformer encoder (Exercise 3 swaps it), and the graph gets a second GRU. Architecture 1, intermediate fusion by concatenation, is M5 (two modalities) and M7 (three). Architecture 2, cross-attention, is not used (Exercise 7). Late fusion (M11) and residual fusion (M12) are added, and so are the gradient-boosting models M1, M2, M9 and M10. |
| 7.5 Training loop | Section 6, `train_net` | The same loop: mini-batches, a check on the validation loans after each epoch, the best checkpoint kept. Early stopping uses the validation AUC instead of the validation loss, and stops after three epochs without improvement. No mixed precision. |
| 7.6 Evaluation | Sections 10 and 11 | ROC AUC as in the chapter, plus the PR-AUC and a paired bootstrap against the benchmark. No confusion matrix: with about 2% events, a threshold of 0.5 classifies almost every loan as good, and a scoring model is judged by its ranking. Section 11 adds loans the model has and has not seen. |

**Before you start:** select a GPU runtime (*Runtime → Change runtime type → T4 GPU*).

Freddie Mac data is used with permission for academic purposes. Do not upload bank or customer data to Colab.

## 0. Setup

Colab already has every package we need: pandas, XGBoost, PyTorch and scikit-learn. We fix the random seed, and we check that the GPU is there.

In [ ]:
import hashlib
import os
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import xgboost as xgb
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore", category=RuntimeWarning)   # means of all-missing windows give NaN, on purpose
SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
THREADS = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
print("CPU cores:", THREADS, "| PyTorch", torch.__version__, "| XGBoost", xgb.__version__)
T_START = time.time()

## 1. The data: one table, one file of monthly records, one file of group counts

We download three files:

- `lab_orig.parquet`: one row per loan, with the data known when the loan was granted, including its 3-digit ZIP code and its seller;
- `lab_perf.parquet`: one row per loan and month, from April 2000 to December 2009, with the monthly record that the servicer reports and the name of the servicer;
- `lab_groups.parquet`: one row per group and month, for three kinds of group (ZIP code, seller, servicer): how many loans of the group reported that month, how many were 30 days late or more, and how many were 90 days late or more. These counts cover every loan in the Freddie Mac sample files of 1999 to 2009, not only the loans of this lab, so that each loan has as many neighbors as the sample allows.

The files keep Freddie Mac's raw codes. We clean them below. We use a modern data format (Parquet) instead of CSV files: the files are smaller and load faster, and each column keeps its type.

In [ ]:
DATA = Path(os.environ.get("CH7_DATA", "data_ch7"))
FILES = {"lab_orig.parquet": "1aQ6x4A5GvEsI-vhMWCwjfVskl-wddLlE",     # Google Drive file ids
         "lab_perf.parquet": "13-HEoXzRYGMj1xHGbH77xcZp4Kve25Y4",
         "lab_groups.parquet": "1-HZpVHJFKgEFfEzq9JHNEGQzAAUOvQ_G"}
DATA.mkdir(exist_ok=True)
for name, file_id in FILES.items():
    if not (DATA / name).exists():
        import gdown
        gdown.download(id=file_id, output=str(DATA / name), quiet=False)

orig = pd.read_parquet(DATA / "lab_orig.parquet")
perf = pd.read_parquet(DATA / "lab_perf.parquet")
groups = pd.read_parquet(DATA / "lab_groups.parquet")
print(f"{len(orig):,} loans, {len(perf):,} monthly records, {len(groups):,} group-months")

Here is one loan's monthly record around its first late payment. `DQ_STATUS` counts the months the payment is late (00 is current, 03 is 90 days late; `RA` means the bank took the property). `CURRENT_UPB` is the unpaid balance. `MOD_FLAG` is `Y` in a month when the loan terms were modified and `P` in the months after. `ZERO_BALANCE_CODE` says why the balance went to zero (01 is a payoff; 02, 03, 09 and 15 end the loan after a credit problem). `SERVICER` is the company that collects the payments; it can change when the servicing is sold.

In [ ]:
months_late = pd.to_numeric(perf.DQ_STATUS.astype(str), errors="coerce")
example = perf.loc[months_late >= 3, "LOAN_ID"].iloc[0]
rec = perf[perf.LOAN_ID == example].reset_index(drop=True)
first_late = (pd.to_numeric(rec.DQ_STATUS.astype(str), errors="coerce") >= 1).idxmax()
rec.iloc[max(0, first_late - 6): first_late + 8]

### Application data

Freddie Mac writes "not available" as a code: 9999 for a missing credit score, 999 for a missing ratio, 99 for a missing count. Used as numbers, these codes would look like very high values, so we turn them into missing values. The same holds for the group names: a seller or servicer with less than 1% of the volume of a quarter appears as `OTHER`, and an unknown ZIP code as `000`. These do not identify a group, so they become missing.

In [ ]:
MISSING = {"CREDIT_SCORE": 9999, "UNITS": 99, "CLTV": 999, "DTI": 999, "LTV": 999,
           "N_BORROWERS": 99, "MI_PCT": 999}
NUM = ["CREDIT_SCORE", "UNITS", "CLTV", "DTI", "ORIG_UPB", "LTV", "ORIG_RATE", "TERM", "N_BORROWERS", "MI_PCT"]
CAT = ["FIRST_TIME_HOMEBUYER", "OCCUPANCY", "CHANNEL", "PPM", "PROPERTY_TYPE", "PURPOSE"]
UNNAMED = ["OTHER", "Other sellers", "Other servicers", ""]

for col, code_value in MISSING.items():
    orig[col] = orig[col].where(orig[col] != code_value)
for col in CAT:
    orig[col] = orig[col].astype("category")
orig["ZIP3"] = orig.ZIP3.astype(str).str[:3].where(lambda z: ~z.isin(["000", "", "nan"]))
orig["SELLER"] = orig.SELLER.where(~orig.SELLER.isin(UNNAMED))
orig[NUM].describe().T.round(2)

## 2. From monthly records to sequences

We put the monthly records into one grid per variable: a row for each loan and a column for each month. Each grid holds one **channel** of the sequence:

| Channel | Meaning |
|---|---|
| `obs` | 1 if the loan has a record that month, 0 before origination |
| `dq` | months late, 0 to 6 (`RA` becomes 6) |
| `upb_ratio` | unpaid balance divided by the original balance |
| `rate_diff` | current interest rate minus the original rate |
| `mod` | 1 if the loan terms were modified this month or before |
| `defer` | 1 if part of the balance is deferred |

These are the raw records, rescaled so that loans of different sizes are comparable. Nothing is summarized: the network sees every month.

We also mark the months with a **default event**: 90 days late or more, the bank taking the property, or a credit-loss zero-balance code.

In [ ]:
def month_index(yyyymm):
    return (np.asarray(yyyymm) // 100) * 12 + np.asarray(yyyymm) % 100 - 1

M0 = int(month_index(200004))
N_MONTHS = int(month_index(200912)) - M0 + 1
loans = pd.Index(orig.LOAN_ID)
r = loans.get_indexer(perf.LOAN_ID)                       # row of each record
m = month_index(perf.PERIOD.values) - M0                  # column of each record

dq_raw = perf.DQ_STATUS.astype(str).values
dq = pd.to_numeric(pd.Series(dq_raw), errors="coerce").values
ra = dq_raw == "RA"
zb = pd.to_numeric(perf.ZERO_BALANCE_CODE.astype(str), errors="coerce").values

CH = ["obs", "dq", "upb_ratio", "rate_diff", "mod", "defer"]
G = {c: np.full((len(loans), N_MONTHS), np.nan, np.float32) for c in CH}
G["obs"][r, m] = 1
G["dq"][r, m] = np.where(ra, 6, np.clip(dq, 0, 6))
G["upb_ratio"][r, m] = perf.CURRENT_UPB.values / orig.ORIG_UPB.values[r]
G["rate_diff"][r, m] = perf.CURRENT_RATE.values - orig.ORIG_RATE.values[r]
G["mod"][r, m] = np.isin(perf.MOD_FLAG.astype(str).values, ["Y", "P"])   # modified now or before
G["defer"][r, m] = perf.DEFERRED_UPB.fillna(0).values > 0
AGE = np.full((len(loans), N_MONTHS), np.nan, np.float32)
AGE[r, m] = perf.LOAN_AGE.astype(float).values
EVENT = np.zeros((len(loans), N_MONTHS), bool)
EVENT[r, m] = (dq >= 3) | ra | np.isin(zb, [2, 3, 9, 15])
print("grid per channel:", G["dq"].shape, "(loans x months)")

## 3. The third modality: the loan's neighbors

Loans are connected. Two loans in the same 3-digit ZIP code share a housing market. Two loans sold by the same lender passed the same underwriting. Two loans handled by the same servicer get the same collection practices. Together these links form a **graph**: each loan is a node, and each group (a ZIP code, a seller, a servicer) connects all its loans. In the language of Chapter 5, it is a bipartite graph of loans and groups, with three kinds of group, so three layers. The ZIP code and the seller of a loan never change; its servicer can change when the servicing is sold, so the servicer layer changes from month to month.

A graph neural network passes messages along the edges: each node collects a summary of its neighbors. We do one round of this by hand, with a fixed mean instead of learned weights. For each loan, each month and each kind of group, we compute three numbers:

- the share of the **other** loans of the group that are 30 days late or more that month,
- the same share for 90 days late or more,
- the number of other loans, on a log scale (how much evidence the shares rest on).

Three details matter.

1. **Leave the loan out.** A loan is part of its own group. If its own delinquency were in the share, the channel would partly copy the target. We subtract the loan itself from the counts.
2. **Shrink small groups.** A share computed on five loans is noisy. We pull it toward the national share with the weight of 50 loans: $(\text{late others} + 50 \cdot \text{national share}) / (\text{others} + 50)$. Then we subtract the national share, so the channel says how much worse or better the neighbors are than the country that month. A rate that rises everywhere at once (as in 2008) says nothing about which loan is riskier; the difference does.
3. **Use last month.** The value of month $m$ is the one of month $m-1$, so that the channel uses only data a bank has at the end of the month.

The result is a grid of nine channels (three per kind of group) for every loan and month. The model reads its last 24 months next to the loan's own payment history.

In [ ]:
KINDS = ["zip3", "seller", "servicer"]
groups = groups[(groups.PERIOD >= 200003) & (groups.PERIOD <= 200912)].copy()
groups["mi"] = month_index(groups.PERIOD.values) - M0 + 1      # column, shifted by one: month m-1 lands in m

# National shares, from the ZIP code layer (every loan with a known ZIP code)
z = groups[groups.level == "zip3"]
nat_n = np.bincount(z.mi, weights=z.n, minlength=N_MONTHS + 1)[:N_MONTHS]
NAT = {"30": np.bincount(z.mi, weights=z.n30, minlength=N_MONTHS + 1)[:N_MONTHS] / np.maximum(nat_n, 1),
       "90": np.bincount(z.mi, weights=z.n90, minlength=N_MONTHS + 1)[:N_MONTHS] / np.maximum(nat_n, 1)}

# Each loan's group in each month: a row number in the count tables, or -1 if unknown
servicer = perf.SERVICER.astype(object).where(~perf.SERVICER.astype(str).isin(UNNAMED))
KEYS, COUNTS, MEMBER = {}, {}, {}
for kind in KINDS:
    lev = groups[groups.level == kind]
    KEYS[kind] = pd.Index(sorted(lev.key.unique()))
    COUNTS[kind] = {}
    for c in ("n", "n30", "n90"):
        t = np.zeros((len(KEYS[kind]) + 1, N_MONTHS + 1), np.float32)   # last row: no group
        t[KEYS[kind].get_indexer(lev.key), lev.mi.values] = lev[c].values
        COUNTS[kind][c] = t[:, :N_MONTHS]
    if kind == "servicer":
        g = np.full((len(loans), N_MONTHS), -1, np.int32)
        g[r, m] = KEYS[kind].get_indexer(servicer)
        g[:, 1:] = g[:, :-1].copy()                                     # the servicer of month m-1
        g[:, 0] = -1
    else:
        g = np.repeat(KEYS[kind].get_indexer(orig["ZIP3" if kind == "zip3" else "SELLER"].astype(object))[:, None],
                      N_MONTHS, axis=1).astype(np.int32)
    MEMBER[kind] = g
    print(f"{kind:9s} {len(KEYS[kind]):4d} groups; loan-months with no known group: "
          f"{np.mean(g[G['obs'] == 1] < 0):.1%}")

# The loan's own state in month m-1, to leave it out of its group
prev = lambda a: np.concatenate([np.zeros((len(a), 1), bool), a[:, :-1]], axis=1)
own_in = prev(G["obs"] == 1)
own30, own90 = prev(np.nan_to_num(G["dq"]) >= 1), prev(np.nan_to_num(G["dq"]) >= 3)


def neighbor_grid(reassign=None):
    """Loans x months x 9 channels. reassign maps each group to another group of its kind (placebo)."""
    out = np.full((len(loans), N_MONTHS, 9), np.nan, np.float32)
    cols = np.arange(N_MONTHS)[None, :]
    for j, kind in enumerate(KINDS):
        g = MEMBER[kind]
        if reassign is not None:
            g = np.where(g >= 0, reassign[kind][np.clip(g, 0, None)], -1)
        row = np.where(g >= 0, g, len(KEYS[kind]))
        self_in = own_in & (g >= 0) & (reassign is None)               # in the placebo the loan is not in the group
        n_others = COUNTS[kind]["n"][row, cols] - self_in
        for w, (c, own) in enumerate((("30", own30), ("90", own90))):
            late_others = COUNTS[kind]["n" + c][row, cols] - (self_in & own)
            share = (late_others + 50 * NAT[c][None, :]) / (n_others + 50)
            out[:, :, 3 * j + w] = np.where(g >= 0, share - NAT[c][None, :], np.nan)
        out[:, :, 3 * j + 2] = np.where(g >= 0, np.log1p(np.clip(n_others, 0, None)), np.nan)
    return out


NB_NAMES = [f"{k}_{s}" for k in KINDS for s in ("ex30", "ex90", "logn")]
NB = neighbor_grid()
print("neighbor grid:", NB.shape, "(loans x months x channels)")

The **placebo** for this modality keeps the form of the neighbor series and breaks their link to the loan. Every ZIP code is replaced by another ZIP code, every seller by another seller and every servicer by another servicer, with one fixed random mapping. The loan then reads the history of a real group, but not of its own. If a model does as well with the placebo, the neighbors add nothing real.

In [ ]:
rng = np.random.default_rng(SEED)
REASSIGN = {}
for kind in KINDS:
    p = rng.permutation(len(KEYS[kind]))
    while (p == np.arange(len(p))).any():                       # no group keeps its own history
        p = rng.permutation(len(KEYS[kind]))
    REASSIGN[kind] = p
NB_PL = neighbor_grid(REASSIGN)

# The excess 90+ share of a loan's ZIP code, for the ZIP codes with the most loans, over time
k = KINDS.index("zip3")
top = pd.Series(MEMBER["zip3"][:, -1]).value_counts().index[:5]
months = pd.period_range("2000-04", "2009-12", freq="M").to_timestamp()
fig, ax = plt.subplots(figsize=(10, 3.5))
for g in top[top >= 0]:
    i = np.where(MEMBER["zip3"][:, -1] == g)[0][0]
    ax.plot(months, NB[i, :, 3 * k + 1], label=f"ZIP {KEYS['zip3'][g]}")
ax.axhline(0, color="black", lw=0.6)
ax.set_ylabel("90+ share minus national share"); ax.legend(ncol=5, fontsize=8)
plt.tight_layout(); plt.show()

### Observation points, target and split

At the end of each quarter $t$ we score every loan that is **current** (not late) at $t$ and at least six months old. The input is the 24 months up to $t$, for the payment history and for the neighbors. The target is 1 if a default event happens in the next 12 months.

The split is by time, so that every training label is known before the test period starts:

- **training**: quarter-ends from March 2002 to December 2006 (their 12-month windows close by December 2007);
- **test**: the four quarter-ends of 2008.

A loan appears once per quarter, so the same loan appears several times. Every split that holds data out (early stopping, folds, the test half) is therefore **by loan**, through a hash of the loan id, so that a loan's quarters never sit on both sides.

For the test we use only half of the loans. We chose the design of this lab, the neighbor channels included, on the other half of the 2008 loans, so this half was never looked at while we chose the models.

In [ ]:
L, H = 24, 12
TRAIN_T = [y * 100 + q for y in range(2002, 2007) for q in (3, 6, 9, 12)]
TEST_T = [200803, 200806, 200809, 200812]

def h01(ids, salt):
    """A number in [0, 1] from each loan id: the same id always gets the same number."""
    return np.array([int(hashlib.md5((salt + s).encode()).hexdigest()[:8], 16) / 0xFFFFFFFF for s in ids])

test_half = h01(loans, "half") < 0.5
points = []                                               # (kind, quarter-end, month column, loan rows)
for kind, quarters in (("train", TRAIN_T), ("test", TEST_T)):
    for t in quarters:
        k = int(month_index(t)) - M0
        ok = (G["obs"][:, k] == 1) & (G["dq"][:, k] == 0) & (AGE[:, k] >= 6) & (G["upb_ratio"][:, k] > 0)
        if kind == "test":
            ok &= test_half
        points.append((kind, t, k, np.where(ok)[0]))

# Fill arrays of the final size directly: building lists and joining them would need twice the memory.
n_rows = sum(len(i) for *_, i in points)
S = np.empty((n_rows, L, len(CH)), np.float32)           # rows x 24 months x 6 channels
SN = np.empty((n_rows, L, 9), np.float32)                # rows x 24 months x 9 neighbor channels
SN_PL = np.empty((n_rows, L, 9), np.float32)
rows, start = [], 0
for kind, t, k, i in points:
    j = slice(start, start + len(i))
    S[j] = np.stack([G[c][i, k - L + 1:k + 1] for c in CH], axis=2)
    SN[j] = NB[i, k - L + 1:k + 1]
    SN_PL[j] = NB_PL[i, k - L + 1:k + 1]
    rows.append(pd.DataFrame({"loan": i, "LOAN_ID": loans[i], "t": t, "kind": kind, "age": AGE[i, k],
                              "y": EVENT[i, k + 1:k + 1 + H].any(axis=1).astype(int)}))
    start += len(i)
D = pd.concat(rows, ignore_index=True)
D = D.join(orig[NUM + CAT].reset_index(drop=True), on="loan")
# Free memory: from here on, only D and the three arrays of windows are needed.
del NB, NB_PL, G, AGE, EVENT, MEMBER, COUNTS, own_in, own30, own90, perf, groups, rows, points

loan_es = h01(loans, "es") < 0.2                          # 20% of training loans for early stopping
loan_fold = (h01(loans, "fold") * 3).astype(int)          # 3 folds, for out-of-fold scores
train = (D.kind == "train").values
tr = train & ~loan_es[D.loan.values]
va = train & loan_es[D.loan.values]
te = (D.kind == "test").values
fold = loan_fold[D.loan.values]
for name, mask in (("training", tr), ("early stopping", va), ("test", te)):
    print(f"{name:15s} {mask.sum():9,} rows, {D.y[mask].sum():5,} events ({D.y[mask].mean():.2%})")

Here is a test loan that defaulted, as the network sees it: three channels of its own payment history, and the excess 90+ share of its ZIP code and of its servicer. The last month on the right is the quarter-end $t$, when the loan was current.

In [ ]:
j = np.where(te & (D.y.values == 1))[0][0]
fig, ax = plt.subplots(1, 4, figsize=(15, 3))
for a, c in zip(ax[:3], ["dq", "upb_ratio", "rate_diff"]):
    a.plot(range(-L + 1, 1), S[j, :, CH.index(c)], marker="o")
    a.set_title(c); a.set_xlabel("months before t")
for name in ("zip3_ex90", "servicer_ex90"):
    ax[3].plot(range(-L + 1, 1), SN[j, :, NB_NAMES.index(name)], marker="o", label=name)
ax[3].axhline(0, color="black", lw=0.6); ax[3].set_title("neighbors"); ax[3].legend(fontsize=8)
ax[3].set_xlabel("months before t")
plt.tight_layout(); plt.show()

## 4. The hand-made features

This is an example of what a bank would compute from the same 24 months: the worst delinquency, the number of late months, the months since the last late payment, the balance paid down, and the current state of the loan. In reality, a behavioral scoring model may construct tens of thousands of variables (thus the technical debt!). A model with these features is the benchmark that the multimodal models must match.

We also summarize the neighbor channels by hand: their value at $t$ and their means over the last 3 and 12 months. A gradient-boosting model with these summaries (M9) tells us whether the graph helps even without a network to read it. We compute the same summaries from the placebo grid too, for its placebo (M10).

In [ ]:
def first_value(a):
    """The first non-missing value of each row."""
    ok = np.isfinite(a)
    v = a[np.arange(len(a)), ok.argmax(axis=1)]
    return np.where(ok.any(axis=1), v, np.nan)

def summaries(S):
    obs = np.nan_to_num(S[:, :, 0])
    dq = np.where(obs == 1, np.nan_to_num(S[:, :, 1]), 0)
    upb = S[:, :, 2]
    late12 = dq[:, -12:] >= 1
    f = pd.DataFrame(index=range(len(S)))
    f["max_dq12"] = dq[:, -12:].max(axis=1)
    f["n_dq12"] = late12.sum(axis=1)
    f["n_dq3"] = (dq[:, -3:] >= 1).sum(axis=1)
    f["since_dq"] = np.where(late12.any(axis=1), late12[:, ::-1].argmax(axis=1), 13)
    f["upb_ratio_t"] = upb[:, -1]
    f["paydown12"] = first_value(upb[:, -12:]) - upb[:, -1]
    f["rate_diff_t"] = S[:, -1, 3]
    f["mod_ever"] = np.nan_to_num(S[:, -12:, 4]).max(axis=1)
    f["defer_t"] = S[:, -1, 5]
    f["max_dq24"] = dq.max(axis=1)
    f["n_dq24"] = (dq >= 1).sum(axis=1)
    f["paydown24"] = first_value(upb) - upb[:, -1]
    return f

def neighbor_summaries(SN):
    f = pd.DataFrame(index=range(len(SN)))
    for c, name in enumerate(NB_NAMES):
        f[f"nb_{name}_t"] = SN[:, -1, c]
        if not name.endswith("logn"):
            f[f"nb_{name}_m3"] = np.nanmean(SN[:, -3:, c], axis=1)
            f[f"nb_{name}_m12"] = np.nanmean(SN[:, -12:, c], axis=1)
    return f

SUMM = summaries(S)
NBS = neighbor_summaries(SN)
NBS_PL = neighbor_summaries(SN_PL).add_suffix("_pl")
D = pd.concat([D, SUMM, NBS, NBS_PL], axis=1)
TAB = NUM + CAT + ["age"]
SUMM_COLS, NB_COLS, NB_COLS_PL = list(SUMM.columns), list(NBS.columns), list(NBS_PL.columns)
D.loc[te, SUMM_COLS].describe().T.round(3)

## 5. Gradient boosting: one modality, then the hand-made features

We fit XGBoost twice: on the application data alone (**M1**), and with the hand-made features of the payment history added (**M2**). M2 is the model a bank would build today. Early stopping uses the held-out 20% of training loans. XGBoost reads the category columns directly (`enable_categorical=True`) and uses the GPU when there is one.

In [ ]:
RESULTS = {}          # model name -> test scores
INFO = {}             # model name -> inputs, fusion, seconds
XGB_DEVICE = "cuda" if DEVICE.type == "cuda" else "cpu"

def fit_xgb(Xtr, ytr, Xva, yva, seed=SEED):
    model = xgb.XGBClassifier(n_estimators=4000, learning_rate=0.03, max_depth=5, min_child_weight=1,
                              subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0, tree_method="hist",
                              enable_categorical=True, device=XGB_DEVICE, eval_metric="auc",
                              early_stopping_rounds=200, random_state=seed, n_jobs=THREADS)
    model.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    model.set_params(device="cpu")       # predict where the data are: pandas frames in CPU memory
    return model

def run_xgb(name, cols, inputs, fusion):
    t0 = time.time()
    model = fit_xgb(D.loc[tr, cols], D.y[tr], D.loc[va, cols], D.y[va])
    RESULTS[name] = model.predict_proba(D.loc[te, cols])[:, 1]
    INFO[name] = (inputs, fusion, time.time() - t0)
    print(f"{name}: test AUC {roc_auc_score(D.y[te], RESULTS[name]):.4f}, "
          f"{model.best_iteration + 1} trees, {time.time() - t0:.0f} s")
    return model

m1 = run_xgb("M1 XGBoost, application", TAB, "application", "none")
m2 = run_xgb("M2 XGBoost, application + hand-made", TAB + SUMM_COLS, "application + hand-made", "features")

## 6. Neural networks on the raw sequences

The networks need numbers without gaps on a common scale. For the table we turn each category into 0/1 columns, fill missing values with the training median (and add a 0/1 column that marks them), and standardize. For the payment sequence, a month before origination gets `obs` = 0 and the training median in the other channels; then each channel is standardized with one mean and one standard deviation, fitted on the training rows. For the neighbor sequence, a missing value means that the loan has no known group of that kind: its excess share becomes 0 (the national share) and its log count 0 (no neighbors). The placebo is scaled with the statistics of the real grid, so that the two differ only in which group they describe.

We move everything to the GPU once. The data are small enough to fit.

In [ ]:
def prep_tab(D, cols, fit):
    X = pd.get_dummies(D[cols], dtype=float)
    miss = X.isna().astype(float).add_suffix("_na")
    miss = miss.loc[:, miss[fit].sum() > 0]
    X = pd.concat([X.fillna(X[fit].median()), miss], axis=1)
    mu, sd = X[fit].mean(), X[fit].std().replace(0, 1)
    return ((X - mu) / sd).values.astype(np.float32)

def prep_seq(S, fit):
    S = S.copy()
    fill = np.nanmedian(S[fit].reshape(-1, S.shape[2]), axis=0)
    for c in range(S.shape[2]):
        v = S[:, :, c]
        v[np.isnan(v)] = 0.0 if c == 0 else fill[c]
    flat = S[fit].reshape(-1, S.shape[2])
    sd = flat.std(axis=0); sd[sd == 0] = 1
    return ((S - flat.mean(axis=0)) / sd).astype(np.float32)

def prep_nb(SN, fit, ref):
    SN = np.nan_to_num(SN, nan=0.0)
    flat = np.nan_to_num(ref[fit], nan=0.0).reshape(-1, SN.shape[2])
    sd = flat.std(axis=0); sd[sd == 0] = 1
    return ((SN - flat.mean(axis=0)) / sd).astype(np.float32)

gpu = lambda a: torch.tensor(a, dtype=torch.float32, device=DEVICE)
X_tab = gpu(prep_tab(D, TAB, tr))
X_seq = gpu(prep_seq(S, tr))
X_nb = gpu(prep_nb(SN, tr, SN))
X_nb_pl = gpu(prep_nb(SN_PL, tr, SN))
del S, SN, SN_PL                                          # the GPU holds the copies we need
Y = D.y.values
ZERO = torch.zeros(len(D), device=DEVICE)
print("table:", tuple(X_tab.shape), "| payments:", tuple(X_seq.shape), "| neighbors:", tuple(X_nb.shape))

One network class covers every design. It has up to three branches:

- a **table branch**, a small multilayer perceptron (MLP) on the application data;
- a **payment branch**, a GRU that reads the 24 months of the loan's own history and returns its last hidden state;
- a **neighbor branch**, a second, smaller GRU that reads the 24 months of the neighbor channels.

A **head** joins whatever branches are present and returns one number, the logit of the score. The head can also add a fixed **offset** to that logit. Residual fusion (Section 9) uses the offset.

In [ ]:
class FusionNet(nn.Module):
    def __init__(self, n_tab, n_ch, n_nb, use_tab=True, use_seq=True, use_nb=False, zero_head=False):
        super().__init__()
        self.tab = nn.Sequential(nn.Linear(n_tab, 64), nn.ReLU(), nn.Dropout(0.1),
                                 nn.Linear(64, 32), nn.ReLU()) if use_tab else None
        self.seq = nn.GRU(n_ch, 64, batch_first=True) if use_seq else None
        self.nb = nn.GRU(n_nb, 32, batch_first=True) if use_nb else None
        width = (32 if use_tab else 0) + (64 if use_seq else 0) + (32 if use_nb else 0)
        self.head = nn.Sequential(nn.Linear(width, 32), nn.ReLU(), nn.Linear(32, 1))
        if zero_head:                      # start as "no correction" (Section 9)
            nn.init.zeros_(self.head[2].weight)
            nn.init.zeros_(self.head[2].bias)

    def forward(self, x_tab, x_seq, x_nb, offset):
        parts = []
        if self.tab is not None:
            parts.append(self.tab(x_tab))
        if self.seq is not None:
            parts.append(self.seq(x_seq)[1][-1])          # last hidden state
        if self.nb is not None:
            parts.append(self.nb(x_nb)[1][-1])
        return self.head(torch.cat(parts, dim=1)).squeeze(1) + offset

One training function serves every network. It trains with mini-batches, checks the AUC on the early-stopping loans after each epoch, keeps the best weights, and stops after three epochs without improvement. It returns the logits for the rows we ask for.

A network depends on its random seed: the starting weights and the order of the mini-batches. In our pilots, the test AUC of the same network moved by 0.003 to 0.006 from one seed to the next. That is as large as some of the effects we test here, so one run per model could mislead us. We train every network with three seeds and average their logits (a small ensemble). This triples the training time, and it makes the comparisons between models much more stable.

In [ ]:
def train_net(fit, stop, predict_for, use_tab=True, use_seq=True, use_nb=False, x_seq=None, x_nb=None,
              offset=None, zero_head=False, max_epochs=15, batch=2048, seed=SEED):
    set_seed(seed)
    x_seq = X_seq if x_seq is None else x_seq
    x_nb = X_nb if x_nb is None else x_nb
    offset = ZERO if offset is None else offset
    net = FusionNet(X_tab.shape[1], x_seq.shape[2], x_nb.shape[2], use_tab, use_seq, use_nb, zero_head).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss()
    y = torch.tensor(Y, dtype=torch.float32, device=DEVICE)

    def logits(idx):
        net.eval(); out = []
        with torch.inference_mode():
            for b in range(0, len(idx), 16384):
                i = idx[b:b + 16384]
                out.append(net(X_tab[i], x_seq[i], x_nb[i], offset[i]).float().cpu())
        net.train()
        return torch.cat(out).numpy()

    fit_i = torch.tensor(np.where(fit)[0], device=DEVICE)
    stop_i = torch.tensor(np.where(stop)[0], device=DEVICE)
    best, best_state, bad = -1.0, None, 0
    for epoch in range(1, max_epochs + 1):
        perm = fit_i[torch.randperm(len(fit_i), device=DEVICE)]
        for b in range(0, len(perm), batch):
            i = perm[b:b + batch]
            opt.zero_grad()
            loss_fn(net(X_tab[i], x_seq[i], x_nb[i], offset[i]), y[i]).backward()
            opt.step()
        auc = roc_auc_score(Y[stop], logits(stop_i))
        if auc > best:
            best, bad = auc, 0
            best_state = {k: v.detach().clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= 3:
                break
    net.load_state_dict(best_state)
    print(f"   {epoch} epochs, best early-stopping AUC {best:.4f}")
    return [logits(torch.tensor(np.where(p)[0], device=DEVICE)) for p in predict_for]

SEEDS = 3

def train_avg(*args, seeds=SEEDS, **kwargs):
    """Train the same network with several seeds and average the logits of each requested set of rows."""
    runs = [train_net(*args, seed=SEED + s, **kwargs) for s in range(seeds)]
    return [np.mean([run[k] for run in runs], axis=0) for k in range(len(runs[0]))]

sigmoid = lambda z: 1 / (1 + np.exp(-z))

def record(name, logit_te, inputs, fusion, t0):
    RESULTS[name] = sigmoid(logit_te)
    INFO[name] = (inputs, fusion, time.time() - t0)
    print(f"{name}: test AUC {roc_auc_score(Y[te], RESULTS[name]):.4f}, {time.time() - t0:.0f} s")

### Each modality alone, then two modalities

**M3** is the MLP on the application data alone. **M4** is the GRU on the payment sequence alone. We train M4 three times, once per fold of training loans: each fold's network scores the loans it did not see. These **out-of-fold** scores are what late fusion needs in Section 9. M4's test score is the average of the three networks, so M4 is already an average of three runs.

**M5** joins the table and payment branches in one network and trains them together, so the GRU learns the sequence features that help *given* the application data. In the terms of the chapter this is **intermediate fusion**: each modality first goes through its own extractor, and the embeddings are concatenated (Architecture 1 of Algorithm 7.4). M2, by contrast, is **early fusion** of featurized data: the hand-made features are computed first and enter one model together with the application data.

**M6** is the **placebo** for the payment history. It is M5 trained and tested with the sequences shuffled across loans within each quarter. The sequences keep their distribution but no longer belong to their loans. If M6 does as well as M5, the payment branch adds nothing real.

In [ ]:
t0 = time.time()
(lt,) = train_avg(tr, va, [te], use_seq=False)
record("M3 MLP, application", lt, "application", "none", t0)

t0 = time.time()
gru_logit = np.full(len(D), np.nan)                       # out-of-fold on training rows
gru_rest = np.zeros(len(D))                               # fold average elsewhere
rest = ~tr
for f in range(3):
    hold, rest_logits = train_net(tr & (fold != f), va, [tr & (fold == f), rest], use_tab=False)
    gru_logit[tr & (fold == f)] = hold
    gru_rest[rest] += rest_logits / 3
gru_logit[rest] = gru_rest[rest]
record("M4 GRU, payments", gru_logit[te], "payments", "none", t0)

t0 = time.time()
(lt,) = train_avg(tr, va, [te])
record("M5 intermediate fusion, MLP + GRU", lt, "application + payments", "intermediate", t0)

rng = np.random.default_rng(SEED)
perm = np.arange(len(D))
for t in D.t.unique():
    i = np.where(D.t.values == t)[0]
    perm[i] = rng.permutation(i)
t0 = time.time()
(lt,) = train_avg(tr, va, [te], x_seq=X_seq[torch.tensor(perm, device=DEVICE)])
record("M6 placebo: M5, shuffled payments", lt, "application + shuffled payments", "intermediate", t0)

## 7. Three modalities: adding the graph

**M7** adds the neighbor branch to M5: the table, the payment history and the graph, fused in one network. **M8** is the placebo for the graph: M7 with every group replaced by another group of its kind (Section 3). **M9** is the hand-made route to the same information: M2 plus the neighbor summaries of Section 4, in XGBoost. **M10** is its placebo: the same summaries, computed from the reassigned groups.

The comparisons to read are M7 against M5 (does the graph add?), M7 against M8 (is the gain real, or would any neighbor history do?), and M9 against M2 and M10 (does the graph help gradient boosting too, and is that gain real?).

In [ ]:
t0 = time.time()
(lt,) = train_avg(tr, va, [te], use_nb=True)
record("M7 intermediate fusion, MLP + GRU + neighbor GRU", lt, "application + payments + graph", "intermediate", t0)

t0 = time.time()
(lt,) = train_avg(tr, va, [te], use_nb=True, x_nb=X_nb_pl)
record("M8 placebo: M7, groups reassigned", lt, "application + payments + reassigned graph", "intermediate", t0)

m9 = run_xgb("M9 XGBoost, M2 + neighbor summaries", TAB + SUMM_COLS + NB_COLS,
             "application + hand-made + hand-made graph", "features")
m10 = run_xgb("M10 placebo: M9, groups reassigned", TAB + SUMM_COLS + NB_COLS_PL,
              "application + hand-made + reassigned hand-made graph", "features")

## 8. Late fusion and residual fusion

In **late fusion** each modality has its own model, and a second model combines their scores. Here the GRU score of M4 becomes one more column for XGBoost (**M11**: application data plus the GRU score, no hand-made features). The GRU score of a training row must come from a network that did not train on that loan. Otherwise the score knows the label, and XGBoost trusts it too much. This is why M4 was trained out of fold.

In **residual fusion** gradient boosting goes first. XGBoost on the application data gives a logit; a network with all three branches adds a **correction** to that logit (**M12**). The head starts at zero, so the network starts as the XGBoost model and moves away only when the data say so. The XGBoost logit of a training row is again out of fold. Residual fusion is not one of the designs in the chapter. It combines a late element (a finished model's score) with intermediate fusion inside the network, which is a form of what the chapter calls hybrid fusion. It also follows the chapter's advice on training: start from the modality that is easiest to learn, here the table, and let the network add the rest.

In [ ]:
D["gru_logit"] = gru_logit
m11 = run_xgb("M11 late fusion, XGBoost + GRU score", TAB + ["gru_logit"], "application + payments", "late")

t0 = time.time()
xgb_logit = np.zeros(len(D))
for f in range(3):
    fit_f, hold = tr & (fold != f), tr & (fold == f)
    model = fit_xgb(D.loc[fit_f, TAB], D.y[fit_f], D.loc[va, TAB], D.y[va])
    xgb_logit[hold] = model.predict(D.loc[hold, TAB], output_margin=True)
    xgb_logit[rest] += model.predict(D.loc[rest, TAB], output_margin=True) / 3
(lt,) = train_avg(tr, va, [te], use_nb=True, offset=gpu(xgb_logit), zero_head=True)
record("M12 residual fusion, XGBoost + three-branch network", lt, "application + payments + graph", "residual", t0)

## 9. Results

We report two measures on the test loans: the AUC, which measures the ranking, and the PR-AUC (average precision), which looks only at how well the riskiest scores catch the defaults. With about 2% events, the PR-AUC moves more than the AUC.

Differences between models come with a 95% interval from a **paired bootstrap by loan**: we draw loans with replacement, keep all their quarters, and recompute both models' AUC on the same draw. An interval that contains zero means the test data cannot tell the two models apart. We give each model's difference from the benchmark M2, and the six differences that answer the questions of this lab.

In [ ]:
def boot_diffs(y, scores, pairs, groups, draws=200, seed=SEED):
    """Paired bootstrap by loan: AUC of model a minus AUC of model b, on the same draws, for each pair."""
    codes, uniq = pd.factorize(groups)
    rng = np.random.default_rng(seed)
    out = {pair: [] for pair in pairs}
    for _ in range(draws):
        w = np.bincount(rng.integers(0, len(uniq), len(uniq)), minlength=len(uniq))[codes]
        auc = {name: roc_auc_score(y, scores[name], sample_weight=w) for name in {n for p in pairs for n in p}}
        for a_, b_ in pairs:
            out[(a_, b_)].append(auc[a_] - auc[b_])
    return {pair: np.percentile(v, [2.5, 97.5]) for pair, v in out.items()}

y_te, loans_te = Y[te], D.LOAN_ID.values[te]
BENCH = "M2 XGBoost, application + hand-made"
name_of = {n.split()[0]: n for n in RESULTS}
QUESTIONS = [("M5", "M3", "the payment history adds to the table"),
             ("M5", "M6", "the payment history, against its placebo"),
             ("M7", "M5", "the graph adds to table + payments"),
             ("M7", "M8", "the graph, against its placebo"),
             ("M9", "M2", "the hand-made graph adds to XGBoost"),
             ("M9", "M10", "the hand-made graph, against its placebo")]
pairs = [(n, BENCH) for n in RESULTS if n != BENCH] + [(name_of[a_], name_of[b_]) for a_, b_, _ in QUESTIONS]
intervals = boot_diffs(y_te, RESULTS, pairs, loans_te)
auc_of = {n: roc_auc_score(y_te, p) for n, p in RESULTS.items()}

table = []
for name, p in RESULTS.items():
    inputs, fusion, secs = INFO[name]
    row = {"model": name, "inputs": inputs, "fusion": fusion, "AUC": auc_of[name],
           "PR-AUC": average_precision_score(y_te, p), "seconds": secs}
    if name != BENCH:
        lo, hi = intervals[(name, BENCH)]
        row["dAUC vs M2"] = auc_of[name] - auc_of[BENCH]
        row["95% interval"] = f"[{lo:+.4f}, {hi:+.4f}]"
    table.append(row)
table = pd.DataFrame(table).set_index("model")
print(f"Test: {te.sum():,} loan-quarters, {y_te.sum():,} events, {len(set(loans_te)):,} loans")
display(table.round(4))

answers = pd.DataFrame([{"question": q, "comparison": f"{a_} - {b_}",
                         "dAUC": auc_of[name_of[a_]] - auc_of[name_of[b_]],
                         "95% interval": "[{:+.4f}, {:+.4f}]".format(*intervals[(name_of[a_], name_of[b_])])}
                        for a_, b_, q in QUESTIONS]).set_index("question")
answers.round(4)

In [ ]:
order = table.index[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(order, table.loc[order, "AUC"], color=["tab:gray" if "placebo" in n else "tab:blue" for n in order])
ax.axvline(table.loc[BENCH, "AUC"], color="tab:red", ls="--", label="M2, hand-made features")
ax.set_xlim(table.AUC.min() - 0.02, table.AUC.max() + 0.02); ax.set_xlabel("test AUC"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()
print(f"Total run time: {(time.time() - T_START) / 60:.1f} minutes")
if DEVICE.type == "cuda":
    print(f"Peak GPU memory held by PyTorch: {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")

### How to read the tables

Check, in this order:

1. **Does each modality add anything?** The payment history adds if M5 beats both M3 (the same network without it) and M6 (its placebo). The graph adds if M7 beats both M5 and M8.
2. **Do the raw modalities match the hand-made features?** Compare the raw-sequence models (M5, M7, M11, M12) with M2.
3. **Does the graph help without a network?** Compare M9 with M2 and with its placebo M10.
4. **Which fusion works?** Compare intermediate (M5, M7), late (M11) and residual (M12) fusion.

### What we found

We ran this notebook on one slice of an H100 GPU (a 2g.20gb MIG instance), not on Colab: 201,995 test loan-quarters with 3,748 events, about 9 minutes in all. Your numbers will differ a little, because training on a GPU is not exactly reproducible. The intervals are from the paired bootstrap.

1. **Both new modalities add.** The payment history lifted the network from 0.778 (M3) to 0.808 (M5), +0.0303 [+0.0253, +0.0349], and its placebo (M6, 0.781) lost almost all of that gain. The graph lifted it again, to 0.818 (M7), +0.0106 [+0.0085, +0.0126] over M5. With the groups reassigned (M8, 0.811), most of that gain disappeared: M7 beat its placebo by +0.0076 [+0.0055, +0.0097]. What helps is the behavior of the loan's own neighbors, not the extra branch.
2. **With the graph, the raw modalities match the hand-made features.** M2 reached 0.817. M5 stayed below it, −0.0097 [−0.0146, −0.0055], but M7 was +0.0009 [−0.0039, +0.0048] from it, an interval that contains zero. Residual fusion (M12), which starts from XGBoost on the application data, was +0.0032 [+0.0001, +0.0070]. Late fusion (M11) was −0.0060 [−0.0074, −0.0043]: a sequence model trained alone, whose score is added afterwards, does not learn what the joint training learns.
3. **The graph also helps gradient boosting.** The hand-made neighbor summaries lifted M2 by +0.0040 [+0.0017, +0.0061] (M9, 0.821, the best AUC of our run). The same summaries computed from reassigned groups (M10) did no better than M2, −0.0022 [−0.0044, +0.0004].

The three-modality network reaches the benchmark with no hand-made features at all, and that is the operational gain of this lab: no feature library to design, validate, document and maintain. The work moves to the data flow of Sections 2 and 3, which must build the sequences and the neighbor channels in the same way in training and in production. The graph adds a requirement of its own: the group counts must cover the whole portfolio every month, not only the loans being scored. Exercise 1 asks why the same graph adds nothing when the test year is 2007.

## 10. Loans the model has seen, and loans it has not

The test quarters are in 2008, but many test loans are older. A loan granted in 2003 was already in the training data at its quarter-ends from 2003 to 2006, with its own application data, history and outcomes. A loan granted in 2007 was never in training. This is not leakage: every training label is known before the test period starts, and a bank scores the loans it already holds every quarter. But the two groups can behave very differently, and a model can look better on loans it has seen than on new ones. Tiukhova et al. (2024) found gaps of this kind when they reported their models separately on seen and unseen customers. In one of their models the AUC fell from 0.79 on seen customers to 0.63 on unseen ones; in another it rose from 0.69 to 0.72.

We split the test loans the same way. A loan is **seen** if any of its quarters was among the rows used to fit the models, and **unseen** otherwise. Loans used only for early stopping count as unseen, because the models never fitted to them. We compare the benchmark (M2) with the best raw-modality model of your run.

In [ ]:
fit_loans = set(D.LOAN_ID[tr])
seen = D.LOAN_ID[te].isin(fit_loans).values
raw_models = [n for n in RESULTS if n.split()[0] in ("M5", "M7", "M11", "M12")]
best_raw = max(raw_models, key=lambda n: roc_auc_score(y_te, RESULTS[n]))
rows = []
for group, mask in (("seen", seen), ("unseen", ~seen)):
    vintage = 2000 + pd.Series(loans_te[mask]).str[1:3].astype(int)
    vintage = vintage.where(vintage < 2050, vintage - 100)
    row = {"loans": "seen in training" if group == "seen" else "never in training",
           "rows": int(mask.sum()), "events": int(y_te[mask].sum()), "event rate": y_te[mask].mean(),
           "vintages (10-90%)": f"{int(vintage.quantile(0.1))}-{int(vintage.quantile(0.9))}"}
    for name in (BENCH, best_raw):
        short = name.split()[0]
        row[f"AUC {short}"] = roc_auc_score(y_te[mask], RESULTS[name][mask])
        row[f"PR-AUC {short}"] = average_precision_score(y_te[mask], RESULTS[name][mask])
    rows.append(row)
print("Best raw-modality model in this run:", best_raw)
pd.DataFrame(rows).set_index("loans").round(4)

Read the table in two ways. **Across the rows**, compare each model on seen and unseen loans. **Across the columns**, check whether the gap between M2 and the raw-modality model is the same in both groups. If the network does relatively worse on unseen loans, it leans more on what it learned about known loans.

Be careful with the cause. Here the two groups are not alike. The unseen loans include the 2007 vintage, granted at the peak of the boom and observed early in its life, and their event rate is clearly higher (see the table). So a gap mixes two things: what the model learned about specific loans, and how different the new vintages are. The lesson for a bank is practical. Report the performance on new loans separately, because a model validated on the loans it was trained on can look better than it will be on the next vintage, and that is the vintage the bank still has to decide on.

In our run, M2 fell from 0.818 on seen loans to 0.805 on unseen ones, and the best raw-modality model (M12) from 0.825 to 0.806. The PR-AUC moved the other way, from 0.113 to 0.143 for M2 and from 0.117 to 0.143 for M12, because the unseen loans have almost twice the event rate (2.5% against 1.3%), and the PR-AUC rises with the event rate. The network's lead over M2 is larger on seen loans (0.006 of AUC) than on unseen ones (0.002), so it may lean slightly more on what it learned about known loans. That difference is small, and we did not compute an interval for it.

## 11. The next step: transactional foundation models

The GRU in this lab learns its sequence features for one portfolio and one target. **Transactional foundation models** take the same idea further. A large transformer is pretrained with self-supervision (for example, by predicting the next event) on the raw event histories of millions of customers: card payments, transfers, app events. The bank then fine-tunes the same model for many tasks, such as scoring, fraud detection and product recommendation, and no task needs its own feature library.

- **E.T.-RNN** (Babaev et al., 2019) is an early step. A GRU on raw card transactions scored credit applications with a ROC AUC of 0.83, against 0.81 for LightGBM with about 7,000 hand-made aggregates (their Table 5).
- **nuFormer** (Braithwaite et al., 2025), at Nubank, pretrains a transformer on transaction sequences and fine-tunes it end to end together with the tabular features, as M5 does here. On credit default prediction it improves the AUC of the production LightGBM model, built on hand-made features, by 1.25% in relative terms. The transformer alone does worse than the tabular features alone. These are the same patterns as in this lab: a small gain in ranking, and a sequence model that needs the table next to it. The operational gain is larger. The authors report that they can remove 90% of the redundant hand-made transaction features with no loss in performance.
- **PRAGMA** (Ostroukhov et al., 2026), at Revolut, pretrains one transformer on banking event sequences and adapts it to credit scoring, fraud, lifetime value and other tasks. For credit scoring it reports ROC-AUC gains of 6% to 12% and PR-AUC gains of 70% to 130% over task-specific models, depending on model size (relative changes, their Table 2). The paper gives no absolute values, because they are commercially sensitive, and does not describe the baselines, so these gains cannot be compared with nuFormer's. Its operational argument is that one model can replace several high-maintenance task models.

The size of the accuracy gain from these models is therefore an open question. The operational argument, fewer features and fewer models to maintain, is the one that both papers make, and it is the one this lab shows. The graph of Section 3 points to the other direction of research: Zandi et al. (2025) learn the neighbor aggregation with a graph neural network on the same Freddie Mac data, with ZIP code and lender layers, instead of fixing it by hand as we did.

## 12. Exercises

1. **A calmer test year.** Change the split to train on the quarter-ends of 2002 to 2005 and test on 2007: `TRAIN_T = [y * 100 + q for y in range(2002, 2006) for q in (3, 6, 9, 12)]` and `TEST_T = [200703, 200706, 200709, 200712]`, and run Sections 3 to 9 again. In our pilot with this split (the same models, with LightGBM instead of XGBoost), the graph added nothing: the three-modality network matched M5, and the neighbor summaries did not improve gradient boosting. Why would the neighbors help in 2008 and not in 2007? Think about when local delinquency starts to differ between ZIP codes, sellers and servicers, how far ahead the target looks, and what the excess shares look like in the figure of Section 3. What does your answer mean for a bank that adds network data to its scoring model in a calm year?
2. **Shorter history.** Set `L = 12` in Section 3 and run the lab again. Does the payment history still add? Do the hand-made features lose more than the networks?
3. **A transformer branch.** Replace the payment GRU with `nn.TransformerEncoder` (two layers, four heads, a learned position embedding), and pool the outputs over the months where `obs` is 1. Is it better than the GRU here?
4. **Which layer of the graph matters?** Train M7 with the three channels of one kind of group set to zero in `X_nb` (for example `X_nb[:, :, 0:3] = 0` removes the ZIP code layer). Which layer carries the signal: geography, the lender, or the servicer?
5. **The crisis deepens.** Score the 2008 quarters one at a time. Do the models lose ranking power as the crisis deepens? Does the graph help more in some quarters than in others?
6. **From score to PD.** These are scores. What would you need to turn M2 or M12 into a PD model for a portfolio? Which data in this lab could you use to calibrate it, and which would you not trust?
7. **Cross-attention.** Architecture 2 of Algorithm 7.4 fuses modalities with cross-attention. Let the table embedding be the query and the 24 monthly GRU outputs the keys and values (`nn.MultiheadAttention`), so that each loan's application data decide which months to attend to. Does it beat the concatenation of M5?

## References

- Babaev, D., Savchenko, M., Tuzhilin, A. and Umerenkov, D. (2019). E.T.-RNN: Applying deep learning to credit loan applications. *KDD '19*, 2183–2190. https://doi.org/10.1145/3292500.3330693
- Braithwaite, D. T., Cavalcanti, M., McEver, R. A., Udagawa, H., Silva, D., Ramanath, R., Meneses, F., Yoshida, A., Wingert, E., Ramos, M., Zanfelice, B. and Gupta, A. (2025). Your spending needs attention: Modeling financial habits with transformers. arXiv:2507.23267.
- Bravo, C., Maldonado, S. and Óskarsdóttir, M. (2026). *Deep Learning in Banking*. Wiley.
- Chen, T. and Guestrin, C. (2016). XGBoost: A scalable tree boosting system. *Proceedings of the 22nd ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 785–794. https://doi.org/10.1145/2939672.2939785
- Cho, K., van Merriënboer, B., Gulcehre, C., Bahdanau, D., Bougares, F., Schwenk, H. and Bengio, Y. (2014). Learning phrase representations using RNN encoder-decoder for statistical machine translation. *EMNLP 2014*, 1724–1734.
- Ostroukhov, M., Mikhailov, R., Iashin, V., Sokolov, A., Akshonov, A., Protasov, V., Goncharov, A., Beloborodov, D., Mullin, V., Enzmann, R. Y., Kolovos, G., Renders, J., Nesterov, P. and Repushko, A. (2026). PRAGMA: Revolut foundation model. arXiv:2604.08649.
- Sculley, D., Holt, G., Golovin, D., Davydov, E., Phillips, T., et al. (2015). Hidden technical debt in machine learning systems. *Advances in Neural Information Processing Systems 28 (NIPS 2015)*, 2503–2511.
- Tiukhova, E., Penaloza, E., Óskarsdóttir, M., Baesens, B., Snoeck, M. and Bravo, C. (2024). INFLECT-DGNN: Influencer prediction with dynamic graph neural networks. *IEEE Access*, 12, 115026–115041. https://doi.org/10.1109/ACCESS.2024.3443533
- Zandi, S., Korangi, K., Óskarsdóttir, M., Mues, C. and Bravo, C. (2025). Attention-based dynamic multilayer graph neural networks for loan default prediction. *European Journal of Operational Research*, 321(2), 586–599. https://doi.org/10.1016/j.ejor.2024.09.025
- Freddie Mac. Single-Family Loan-Level Dataset. https://www.freddiemac.com/research/datasets/sf-loanlevel-dataset